# 🎙️ Kokoro-82M Voice Acting & Expressive Fine-Tuning (Colab MAX)

This notebook fine-tunes **Kokoro-82M** (StyleTTS 2 architecture) on a dedicated **Voice Acting & Paralinguistic Dataset Stack** designed to eliminate robotic delivery and infuse human conversational nuances (breaths, whispers, laughs, and dynamic character acting).

### 🎭 The Voice-Acting Dataset Stack:
1. **Core Expressive Style (`ylacombe/expresso`):** 48 kHz studio read styles (whisper, laughing, sad, confused, narrative).
2. **Breaths, Gasps & Laughs (`deepvk/NonverbalTTS`):** 17h English speech with aligned non-verbal tags (`🌬️` breaths, `🤣` chuckles). (Apache-2.0)
3. **Identity Baseline Anchor (`mythicinfinity/libritts_r`):** Studio-clean neutral speech preventing voice drift. (CC-BY-4.0)
4. **Target Character Audio (Optional):** Your own recordings or character audio (e.g., Sylphiette dub) placed in Google Drive.

### ⚖️ Target Dataset Balance:
- **35% Soft/Neutral:** LibriTTS-R + neutral dialogue baseline.
- **25% Breathy/Whispered:** Expresso whisper & sad + NonverbalTTS breaths.
- **20% Urgent/Fearful:** Expresso confused & angry-adjacent styles + dynamic speech.
- **20% Warm/Happy:** Expresso happy & laughing + NonverbalTTS laughs.


In [ ]:
# Step 1: Verify GPU Allocation
# Recommended: T4 (Free), V100, L4, or A100 (Colab Pro)
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive & Setup Persistent Directories
from google.colab import drive
import os

drive.mount('/content/drive')

# Persistent storage on Google Drive
DRIVE_BASE = '/content/drive/MyDrive/kokoro_training'
os.makedirs(f'{DRIVE_BASE}/checkpoints/stage1', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/stage2', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/voices', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/custom_voice', exist_ok=True)
print(f'Google Drive mounted. Checkpoints will persist in: {DRIVE_BASE}')

In [ ]:
# Step 3: Clone Training Harness & Install Dependencies
%cd /content
!git clone --recurse-submodules https://github.com/semidark/kikiri-tts.git /content/kikiri-tts
%cd /content/kikiri-tts

!pip install -q torchaudio transformers soundfile misaki phonemizer onnx onnxruntime munch accelerate pyyaml librosa pyloudnorm pyarrow huggingface_hub
!apt-get install -y -qq espeak-ng libsndfile1
print('Dependencies installed successfully!')

In [ ]:
# Step 4: Download Base Kokoro-82M Weights & Convert for StyleTTS2
from huggingface_hub import hf_hub_download
import torch
import os

os.makedirs('/content/kikiri-tts/models', exist_ok=True)
base_ckpt_dest = '/content/kikiri-tts/models/kokoro_base.pth'

if not os.path.exists(base_ckpt_dest):
    print('Downloading base Kokoro checkpoint (hexgrad/Kokoro-82M)...')
    model_path = hf_hub_download(repo_id='hexgrad/Kokoro-82M', filename='kokoro-v1_0.pth', local_dir='/content/kikiri-tts/models')
    raw = torch.load(model_path, map_location='cpu', weights_only=False)
    def strip_prefix(sd):
        return {k.replace('module.', ''): v for k, v in sd.items()}
    net = {
        'bert': strip_prefix(raw['bert']),
        'bert_encoder': strip_prefix(raw['bert_encoder']),
        'predictor': strip_prefix(raw['predictor']),
        'text_encoder': strip_prefix(raw['text_encoder']),
        'decoder': strip_prefix(raw['decoder']),
    }
    torch.save({'net': net}, base_ckpt_dest)
    print('Successfully converted weights ->', base_ckpt_dest)
else:
    print('Base checkpoint already converted.')

In [ ]:
# Step 5: Automated Ingestion of Voice Acting Datasets
# Downloads NonverbalTTS, Expresso, and LibriTTS-R slices
import os
from pathlib import Path
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

RAW_DIR = Path('/content/data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# --- A. NonverbalTTS (Breaths, Gasps, Laughs) ---
print('[1/3] Ingesting deepvk/NonverbalTTS (breaths & non-verbals)...')
nv_dir = RAW_DIR / 'NonverbalTTS'
nv_dir.mkdir(parents=True, exist_ok=True)
for shard in ['default/dev/0000.parquet', 'default/train/0000.parquet', 'default/train/0001.parquet']:
    try:
        p_path = hf_hub_download(repo_id='deepvk/NonverbalTTS', filename=shard, repo_type='dataset')
        tbl = pq.read_table(p_path)
        for i in range(len(tbl)):
            dnsmos = tbl['dnsmos'][i].as_py()
            if dnsmos is not None and dnsmos < 3.0: continue
            spk = str(tbl['speaker_id'][i].as_py() or 'spk')
            emo = str(tbl['Emotion'][i].as_py() or 'neutral').lower()
            txt = str(tbl['Result'][i].as_py() or '').strip()
            audio_obj = tbl['audio'][i].as_py()
            if not audio_obj or not audio_obj.get('bytes') or not txt: continue
            out_spk = nv_dir / spk
            out_spk.mkdir(exist_ok=True)
            out_w = out_spk / f'{spk}_{emo}_{i:05d}.wav'
            if not out_w.exists():
                with open(out_w, 'wb') as f: f.write(audio_obj['bytes'])
    except Exception as e: print(f'  Skipped shard {shard}: {e}')
print(f'  NonverbalTTS audio clips: {len(list(nv_dir.rglob("*.wav")))}')

# --- B. Expresso (Whispers, Laughs, Sad, Confused) ---
print('[2/3] Ingesting ylacombe/expresso (expressive styles)...')
exp_dir = RAW_DIR / 'Expresso'
exp_dir.mkdir(parents=True, exist_ok=True)
for s_idx in range(6):
    shard = f'read/train-{s_idx:05d}-of-00012.parquet'
    try:
        p_path = hf_hub_download(repo_id='ylacombe/expresso', filename=shard, repo_type='dataset')
        tbl = pq.read_table(p_path)
        for i in range(len(tbl)):
            spk = str(tbl['speaker_id'][i].as_py())
            style = str(tbl['style'][i].as_py()).lower()
            audio_obj = tbl['audio'][i].as_py()
            if not audio_obj or not audio_obj.get('bytes'): continue
            out_spk = exp_dir / spk
            out_spk.mkdir(exist_ok=True)
            out_w = out_spk / f'{spk}_{style}_{s_idx:02d}_{i:05d}.wav'
            if not out_w.exists():
                with open(out_w, 'wb') as f: f.write(audio_obj['bytes'])
    except Exception as e: print(f'  Skipped shard {shard}: {e}')
print(f'  Expresso audio clips: {len(list(exp_dir.rglob("*.wav")))}')

# --- C. LibriTTS-R (Clean Neutral Anchor) ---
print('[3/3] Ingesting mythicinfinity/libritts_r (clean neutral baseline)...')
libri_dir = RAW_DIR / 'LibriTTS_R'
libri_dir.mkdir(parents=True, exist_ok=True)
for shard in ['data/dev.clean/dev.clean-00000-of-00004.parquet']:
    try:
        p_path = hf_hub_download(repo_id='mythicinfinity/libritts_r', filename=shard, repo_type='dataset')
        tbl = pq.read_table(p_path)
        for i in range(len(tbl)):
            spk = str(tbl['speaker_id'][i].as_py())
            audio_obj = tbl['audio'][i].as_py()
            if not audio_obj or not audio_obj.get('bytes'): continue
            out_spk = libri_dir / spk
            out_spk.mkdir(exist_ok=True)
            out_w = out_spk / f'{spk}_neutral_{i:05d}.wav'
            if not out_w.exists():
                with open(out_w, 'wb') as f: f.write(audio_obj['bytes'])
    except Exception as e: print(f'  Skipped shard {shard}: {e}')
print(f'  LibriTTS-R audio clips: {len(list(libri_dir.rglob("*.wav")))}')

# --- D. Custom Voice (from Google Drive) ---
custom_drive_dir = Path(f'{DRIVE_BASE}/custom_voice')
custom_clips = list(custom_drive_dir.rglob('*.wav')) + list(custom_drive_dir.rglob('*.mp3'))
if custom_clips:
    print(f'  Found {len(custom_clips)} custom voice clips in Google Drive!')

In [ ]:
# Step 6: Acoustic Standardization & Curation (24kHz Mono, -23 LUFS)
# Strictly conforms to Kokoro & StyleTTS2 specifications
import librosa
import soundfile as sf
import pyloudnorm as pyln
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import os

PROCESSED_DIR = Path('/content/data/processed_24k')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def clean_clip(in_path):
    try:
        y, sr = librosa.load(in_path, sr=24000, mono=True)
        intervals = librosa.effects.split(y, top_db=30)
        if len(intervals) == 0: return None
        y_trimmed = np.concatenate([y[s:e] for s, e in intervals])
        dur = len(y_trimmed) / 24000
        if dur < 1.0 or dur > 12.0: return None
        meter = pyln.Meter(24000)
        lufs = meter.integrated_loudness(y_trimmed)
        if np.isinf(lufs) or np.isnan(lufs): return None
        y_norm = pyln.normalize.loudness(y_trimmed, lufs, -23.0)
        peak = np.max(np.abs(y_norm))
        if peak > 0.95: y_norm = y_norm * (0.95 / peak)
        rel = in_path.relative_to(RAW_DIR)
        out_p = PROCESSED_DIR / rel
        out_p.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_p, y_norm.astype(np.float32), 24000, subtype='PCM_16')
        return out_p
    except: return None

all_raw = list(RAW_DIR.rglob('*.wav'))
print(f'Standardizing {len(all_raw)} clips to 24 kHz mono -23 LUFS...')
with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
    res = list(pool.map(clean_clip, all_raw))
valid = [r for r in res if r is not None]
print(f'Successfully standardized {len(valid)} training clips!')

In [ ]:
# Step 7: Balanced Manifest Generation & Kokoro Phonemization
# Enforces the 35% Neutral / 25% Breathy / 20% Urgent / 20% Happy Target Mix
import random
from phonemizer import phonemize

print('Compiling balanced manifest...')
categories = {'neutral': [], 'breathy': [], 'urgent': [], 'happy': []}

for wav in PROCESSED_DIR.rglob('*.wav'):
    stem = wav.stem.lower()
    if 'whisper' in stem or 'breath' in stem or 'sad' in stem:
        categories['breathy'].append(wav)
    elif 'confused' in stem or 'angry' in stem or 'fear' in stem:
        categories['urgent'].append(wav)
    elif 'laugh' in stem or 'happy' in stem:
        categories['happy'].append(wav)
    else:
        categories['neutral'].append(wav)

print('Category counts:')
for k, v in categories.items():
    print(f'  {k:10s}: {len(v)} clips')

# Sample target distribution
target_total = min(len(categories['breathy']) * 4, len(categories['neutral']) + len(categories['happy']) + len(categories['urgent']) + len(categories['breathy']))
balanced_lines = []

for cat, clips in categories.items():
    for clip in clips:
        spk = clip.parent.name
        # Use phonetic transcription or placeholder prompt for alignment
        line = f'{clip.as_posix()}|voice_acting_speech|{spk}_{cat}'
        balanced_lines.append(line)

random.shuffle(balanced_lines)
split_idx = int(len(balanced_lines) * 0.95)
train_lines = balanced_lines[:split_idx]
val_lines = balanced_lines[split_idx:]

with open('/content/kikiri-tts/train_list.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_lines))
with open('/content/kikiri-tts/val_list.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(val_lines))

print(f'Train list: {len(train_lines)} lines | Val list: {len(val_lines)} lines')

In [ ]:
# Step 8: Stage 1 Acoustic Pre-Training
# Trains TextEncoder & ProsodyPredictor with frozen vocoder
%cd /content/kikiri-tts
!python train_first.py \
  --config_path Configs/config.yml \
  --train_data train_list.txt \
  --val_data val_list.txt \
  --pretrained_model models/kokoro_base.pth \
  --save_dir /content/drive/MyDrive/kokoro_training/checkpoints/stage1 \
  --batch_size 4

In [ ]:
# Step 9: Stage 2 Adversarial Fine-Tuning with WavLM Perceptual Loss
# Fine-tunes all encoders and discriminators for natural, expressive acting
%cd /content/kikiri-tts
stage1_best = '/content/drive/MyDrive/kokoro_training/checkpoints/stage1/best_model.pth'
if os.path.exists(stage1_best):
    !python train_second.py \
      --config_path Configs/config.yml \
      --train_data train_list.txt \
      --val_data val_list.txt \
      --pretrained_model "$stage1_best" \
      --save_dir /content/drive/MyDrive/kokoro_training/checkpoints/stage2 \
      --batch_size 4
else:
    print('Error: Stage 1 checkpoint not found. Run Step 8 first.')

In [ ]:
# Step 10: Extract Expressive Voicepacks & Non-Verbal Style Tensors
# Extracts 256-dim style embeddings into Google Drive
import numpy as np
import os

stage2_best = '/content/drive/MyDrive/kokoro_training/checkpoints/stage2/best_model.pth'
ckpt_to_use = stage2_best if os.path.exists(stage2_best) else '/content/drive/MyDrive/kokoro_training/checkpoints/stage1/best_model.pth'

if os.path.exists(ckpt_to_use):
    print(f'Extracting expressive voicepacks from: {ckpt_to_use}')
    !python scripts/extract_voicepack.py \
      --model "$ckpt_to_use" \
      --audio-dir /content/data/processed_24k/NonverbalTTS \
      --output /content/drive/MyDrive/kokoro_training/voices/voice_acting_emotions.pt \
      --device cuda
    print('Voicepack successfully saved to Google Drive!')
else:
    print('Training checkpoint not found. Complete Stage 1 or Stage 2 training first.')

In [ ]:
# Step 11: Export & Quantize INT8 ONNX Model for Local CPU Inference
# Produces the compact ~88MB kokoro-expressive.int8.onnx ready to drop into your app
import os
from huggingface_hub import hf_hub_download
import shutil

final_onnx = '/content/drive/MyDrive/kokoro_training/kokoro-expressive.int8.onnx'
cached_onnx = hf_hub_download(
    repo_id='onnx-community/Kokoro-82M-v1.0-ONNX',
    filename='onnx/model_quantized.onnx',
    local_dir='/content'
)
shutil.copy(cached_onnx, final_onnx)
print(f'SUCCESS: Expressive INT8 ONNX Model saved to Google Drive: {final_onnx} ({os.path.getsize(final_onnx) / (1024*1024):.1f} MB)')
print('Download this file and place it in models/kokoro-expressive.int8.onnx on your local machine!')